### Setup & Dynamic Cleaning Utility
This cell initializes the environment, sets the visual theme, and defines a robust cleaning function.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Apply a clean, professional aesthetic
sns.set_theme(style="whitegrid", palette="muted")

def clean_plot_data(df, columns):
    """
    Dynamically cleans the DataFrame right before plotting.
    Replaces np.inf with NaN, then drops NaNs only in the required columns.
    """
    # Create a copy of only the required columns to save memory
    plot_df = df[columns].copy()
    plot_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return plot_df.dropna()

# Identify the target column (Label or attack_type based on the dataset state)
# We will assume 'Normal' indicates benign traffic.
target_col = 'Label' # Change to 'attack_type' if needed

### Core Anomaly Scatter Plot
This plot maps `Flow Duration` vs `Flow Bytes/s` using log scales.

In [ ]:
# Select columns and clean data
scatter_cols = ['Flow Duration', 'Flow Bytes/s', target_col]
df_scatter = clean_plot_data(df, scatter_cols)

# Separate normal and anomalous traffic for distinct styling
normals = df_scatter[df_scatter[target_col] == 'Normal']
anomalies = df_scatter[df_scatter[target_col] != 'Normal']

plt.figure(figsize=(10, 6))

# Plot Normal traffic (subtle, semi-transparent)
plt.scatter(normals['Flow Duration'], normals['Flow Bytes/s'], 
            alpha=0.3, label='Normal Traffic', color='#3498db', edgecolors='none', s=20)

# Plot Zombie API / Anomalous traffic (bold, high contrast)
plt.scatter(anomalies['Flow Duration'], anomalies['Flow Bytes/s'], 
            alpha=0.85, label='Anomaly (Zombie API)', color='#e74c3c', marker='x', s=50)

# Logarithmic scaling to handle massive network flow outliers gracefully
plt.xscale('log')
plt.yscale('log')

plt.title('Core Anomaly Detection: Flow Duration vs. Bytes/s', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Flow Duration (Log Scale)', fontsize=12)
plt.ylabel('Flow Bytes/s (Log Scale)', fontsize=12)

# Improve legend presentation
plt.legend(title='Traffic Class', frameon=True, shadow=True)
plt.tight_layout()
plt.show()

### Behavioral Distribution (Engineered Features)
Compares normal vs anomalous behavior across engineered features.

In [ ]:
# Features to plot in our grid
engineered_features = ['packet_ratio', 'bytes_per_packet', 'burstiness']
grid_cols = engineered_features + [target_col]
df_grid = clean_plot_data(df, grid_cols)

# Create a 1x3 subplot grid
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Custom palette ensuring red aligns with anomalies and blue with normal
custom_palette = {'Normal': '#3498db'}
# Dynamically add anomaly class colors to palette
anomaly_classes = df_grid[target_col].unique()
for ac in anomaly_classes:
    if ac != 'Normal':
        custom_palette[ac] = '#e74c3c'

for i, feature in enumerate(engineered_features):
    # Using boxplot to show median, quartiles, and distribution spread
    sns.boxplot(data=df_grid, x=target_col, y=feature, ax=axes[i], 
                palette=custom_palette, showfliers=False) # Hide extreme outliers for scale readability
    
    axes[i].set_title(f'Distribution of {feature}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Traffic Type', fontsize=11)
    axes[i].set_ylabel(feature, fontsize=11)
    
    # Rotate x-labels if there are multiple anomaly types
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Behavioral Feature Distributions: Normal vs. Zombie API Patterns', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

### Timing & Frequency Analysis (KDE)
Generates an overlapping density plot for request timings.

In [ ]:
# Fallback to Flow IAT Mean if time_between_requests isn't available
timing_feature = 'time_between_requests' if 'time_between_requests' in df.columns else 'Flow IAT Mean'
timing_cols = [timing_feature, target_col]

df_timing = clean_plot_data(df, timing_cols)

plt.figure(figsize=(10, 6))

# Plot overlapping Kernel Density Estimates
sns.kdeplot(data=df_timing, x=timing_feature, hue=target_col, 
            fill=True, common_norm=False, palette=custom_palette, alpha=0.5, linewidth=2)

# Using log scale for X as inter-arrival times often span milliseconds to hours
plt.xscale('log')

plt.title(f'Timing Analysis: Pacing & Frequency ({timing_feature})', fontsize=15, fontweight='bold', pad=15)
plt.xlabel(f'{timing_feature} (Log Scale)', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)

# Contextual comment for presentation:
# Narrow, sharp peaks in anomalies often indicate hard-coded bot pacing (Zombie APIs),
# whereas normal traffic usually presents a smoother, wider distribution.

plt.tight_layout()
plt.show()